# 02 - Feature Engineering & Temporal State Construction
## PRISM: Building State Representations $S_t \in \mathbb{R}^{D}$

This notebook demonstrates the end-to-end data transformation pipeline:
1. Flow-level cleaning, log transforms, and z-score normalization (`FlowExtractor`)
2. Packet-level Scapy extraction and port-scanning heuristics (`PacketExtractor`)
3. 5-tuple + time window merging (`FeatureMerger`)
4. Temporal windowing into state vectors $S_t$ (`StateBuilder`)

In [ ]:
import numpy as np
import pandas as pd
from src.data.flow_extractor import FlowExtractor
from src.data.packet_extractor import _is_sequential
from src.data.state_builder import StateBuilder
from src.data.feature_merger import FeatureMerger

### 1. Sequential vs. Randomized Port Scan Detection Heuristic

In [ ]:
ports_sequential = [1000, 1001, 1002, 1003, 1004, 1005]
ports_random = [80, 443, 8080, 22, 53, 9000]

print('Sequential probe detected:', _is_sequential(ports_sequential, min_length=5))
print('Random probe detected:', _is_sequential(ports_random, min_length=5))

### 2. Log-Transform for Highly Skewed NetFlow Features

In [ ]:
durations = np.array([10, 500, 10000, 500000, 100000000.0])
log_durations = np.log1p(durations)
df_demo = pd.DataFrame({'Raw Duration (us)': durations, 'Log1p Duration': log_durations})
df_demo

### 3. Verify Final State Vector Dimensions

In [ ]:
builder = StateBuilder(window_size_seconds=30, mode='vector')
print(f'StateBuilder configured with window_size={builder.window_size_seconds}s')